In [1]:
import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'

# ELO configuration
ELO_K = 20  # ELO K-factor
INITIAL_ELO = 1500
team_elo = {}  # Global dictionary to track ELO ratings

In [2]:
game_df = pd.read_csv(f'{DATA_DIR}/game.csv')
print(f"Loaded game.csv: {len(game_df)} rows")

Loaded game.csv: 65698 rows


In [3]:
def drop_dups(game_df):
    return game_df.drop_duplicates(subset=['game_id'], keep="first")

game_df = drop_dups(game_df)
print(f"Rows after dropping duplicates: {len(game_df)}")

Rows after dropping duplicates: 65642


In [4]:
def prepare_base_data(game_df):
    # Отсортируем датасет по дате, поскольку важна последовательность игр - чтобы в тест не протекли данные с трейна
    game_df['game_date'] = pd.to_datetime(game_df['game_date'])
    game_df = game_df.sort_values('game_date').reset_index(drop=True)
    
    # home_win - эту фичу мы будем предсказывать
    game_df['home_win'] = (game_df['wl_home'] == 'W').astype(int)
    return game_df

game_df = prepare_base_data(game_df)

In [5]:
def extract_basic_features(game_df):
    cols_to_copy = [
        "game_id", "game_date", "season_id",
        "team_id_home", "team_id_away", "home_win",
    
        "fg_pct_home", "fg_pct_away", # процент обычных попаданий в кольцо
        "fg3_pct_home", "fg3_pct_away", # процент 3ех очковых попаданий в кольцо
        "ft_pct_home", "ft_pct_away", # процент попаданий штрафных

        # NB: подбор - это когда игрок завладевает мечом после промаха по кольцу. Если игрок из атакующей команды - то это атакующий подбор 
        "reb_home", "reb_away", # количество подборов всего
        "oreb_home", "oreb_away", # количество атакующих подборов

        "ast_home", "ast_away", # кол-во передач

        "stl_home", "stl_away", # кол-во перехватов меча у противкника
        "blk_home", "blk_away", # кол-во блоков защитника при броске противника

        "tov_home", "tov_away", # команда теряет мяч

        "plus_minus_home", "plus_minus_away",
    ]
    
    features = game_df.loc[:, cols_to_copy].copy()

    # features['is_home'] = 1
    features['reb_diff'] = features['reb_home'] - features['reb_away']
    features['ast_diff'] = features['ast_home'] - features['ast_away']
    features['tov_diff'] = features['tov_home'] - features['tov_away']
    
    return features

features = extract_basic_features(game_df)

In [6]:
def calculate_elo_ratings(features, team_elo, elo_k=ELO_K, initial_elo=INITIAL_ELO):
    features['home_elo_before'] = 0.0
    features['away_elo_before'] = 0.0
    features['elo_diff'] = 0.0
    
    for team_id in pd.concat([features['team_id_home'], features['team_id_away']]).unique():
        if pd.notna(team_id):
            team_elo[team_id] = initial_elo
    
    for idx, row in features.iterrows():
        home_team = row['team_id_home']
        away_team = row['team_id_away']
        home_win = row['home_win']
        
        if pd.isna(home_team) or pd.isna(away_team):
            continue
        
        home_elo = team_elo.get(home_team, initial_elo)
        away_elo = team_elo.get(away_team, initial_elo)
        
        features.at[idx, 'home_elo_before'] = home_elo
        features.at[idx, 'away_elo_before'] = away_elo
        features.at[idx, 'elo_diff'] = home_elo - away_elo
        
        # https://ru.wikipedia.org/wiki/Рейтинг_Эло#Вычисление_рейтинга_Эло
        expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
        
        home_elo_new = home_elo + elo_k * (home_win - expected_home)
        away_elo_new = away_elo + elo_k * ((1 - home_win) - (1 - expected_home))
        
        team_elo[home_team] = home_elo_new
        team_elo[away_team] = away_elo_new
    
    return features

features = calculate_elo_ratings(features, team_elo)

In [7]:
def calculate_rolling_stats(features, windows=[5, 10, 20]):
    df_sorted = features.sort_values(['team_id_home', 'game_date']).copy()
    
    for window in windows:
        print(f"  Processing {window}-game window...")
        print(len(df_sorted))
        
        # Home team rolling stats
        df_sorted_groupby = df_sorted.groupby('team_id_home')
        for stat in ['fg_pct', 'fg3_pct', 'reb', 'ast', 'tov']:
            col_name = f'{stat}_home'
            if col_name in df_sorted.columns:
                df_sorted[f'home_{stat}_L{window}'] = df_sorted_groupby[col_name].transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
                )
            else:
                print(f'Missing column {col_name}')
        
        # Away team rolling stats
        df_sorted_away = features.sort_values(['team_id_away', 'game_date']).copy()
        df_sorted_away_groupby = df_sorted_away.groupby('team_id_away')
        for stat in ['fg_pct', 'fg3_pct', 'reb', 'ast', 'tov']:
            col_name = f'{stat}_away'
            if col_name in df_sorted_away.columns:
                df_sorted_away[f'away_{stat}_L{window}'] = df_sorted_away_groupby[col_name].transform(
                    lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
                )
            else: 
                print(f'Missing column {col_name}')
        
        # Merge away stats back
        away_cols = [c for c in df_sorted_away.columns if f'_L{window}' in c and 'away_' in c]
        old_size = len(df_sorted)
        df_sorted = df_sorted.merge(
            df_sorted_away[['game_id'] + away_cols],
            on='game_id',
            how='left'
        )
        if old_size != len(df_sorted):
            print(f'be carefull - amount of rows changed  from {old_size} to {len(df_sorted)}')
        
        
        # Calculate win rate in last N games
        df_sorted[f'home_win_rate_L{window}'] = df_sorted.groupby('team_id_home')['home_win'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
        )
        
        df_sorted_away[f'away_win_rate_L{window}'] = df_sorted_away.groupby('team_id_away')['home_win'].transform(
            lambda x: (1 - x).rolling(window=window, min_periods=1).mean().shift(1)
        )
        
        df_sorted = df_sorted.merge(
            df_sorted_away[['game_id', f'away_win_rate_L{window}']],
            on='game_id',
            how='left'
        )
    
    # Restore original order
    features = df_sorted.sort_values('game_date').reset_index(drop=True)
    
    return features

features = calculate_rolling_stats(features)

  Processing 5-game window...
65642
  Processing 10-game window...
65642
  Processing 20-game window...
65642


In [8]:
def calculate_head_to_head(features):
    features['h2h_home_wins'] = 0
    features['h2h_total_games'] = 0
    features['h2h_home_win_rate'] = 0.5
    
    h2h_records = {}
    
    for idx, row in features.iterrows():
        home_team = row['team_id_home']
        away_team = row['team_id_away']
        
        if pd.isna(home_team) or pd.isna(away_team):
            continue
        
        matchup_key = tuple(sorted([home_team, away_team]))
        
        if matchup_key not in h2h_records:
            h2h_records[matchup_key] = {'home_wins': 0, 'total': 0}
        
        h2h = h2h_records[matchup_key]
        features.at[idx, 'h2h_home_wins'] = h2h['home_wins']
        features.at[idx, 'h2h_total_games'] = h2h['total']
        
        if h2h['total'] > 0:
            features.at[idx, 'h2h_home_win_rate'] = h2h['home_wins'] / h2h['total']
        
        h2h['total'] += 1
        if row['home_win'] == 1:
            h2h['home_wins'] += 1
    
    return features

features = calculate_head_to_head(features)

In [9]:
def calculate_rest_days(features):
    features['home_rest_days'] = 0
    features['away_rest_days'] = 0
    features['rest_advantage'] = 0
    
    last_game = {}
    
    for idx, row in features.iterrows():
        home_team = row['team_id_home']
        away_team = row['team_id_away']
        game_date = row['game_date']
        
        if pd.isna(home_team) or pd.isna(away_team):
            continue
        
        # Calculate rest days
        if home_team in last_game:
            home_rest = (game_date - last_game[home_team]).days
            features.at[idx, 'home_rest_days'] = home_rest
        
        if away_team in last_game:
            away_rest = (game_date - last_game[away_team]).days
            features.at[idx, 'away_rest_days'] = away_rest
        
        # Rest advantage (positive means home team is more rested)
        features.at[idx, 'rest_advantage'] = features.at[idx, 'home_rest_days'] - features.at[idx, 'away_rest_days']
        
        # Update last game dates
        last_game[home_team] = game_date
        last_game[away_team] = game_date
    
    print(f"Rest days calculated. Average rest: {features['home_rest_days'].mean():.1f} days\n")
    return features

# Calculate rest days
features = calculate_rest_days(features)

Rest days calculated. Average rest: 5.3 days



In [10]:
def add_season_features(features):
    # Extract season year
    features['season_year'] = features['season_id'].astype(str).str[:4].astype(float)
    
    # Calculate games into season for each team
    features['home_games_played'] = features.groupby(['team_id_home', 'season_id']).cumcount()
    features['away_games_played'] = features.groupby(['team_id_away', 'season_id']).cumcount()
    
    # Season progress (0 to 1)
    features['season_progress'] = features['home_games_played'] / 82.0  # NBA regular season is 82 games
    
    return features

features = add_season_features(features)

In [11]:
def create_differential_features(features):
    # Rolling stats differentials
    for window in [5, 10, 20]:
        for stat in ['fg_pct', 'fg3_pct', 'reb', 'ast', 'tov', 'win_rate']:
            home_col = f'home_{stat}_L{window}'
            away_col = f'away_{stat}_L{window}'
            
            if home_col in features.columns and away_col in features.columns:
                features[f'{stat}_diff_L{window}'] = features[home_col] - features[away_col]
    
    return features

features = create_differential_features(features)

In [12]:
# Remove rows with too many missing values (early games)
initial_rows = len(features)
features = features.dropna(subset=['elo_diff', 'home_win_rate_L5'])
print(f"Removed {initial_rows - len(features)} rows with insufficient history")
print(f"Final dataset: {len(features)} games\n")


Removed 63 rows with insufficient history
Final dataset: 65579 games



In [13]:
output_path = 'features_engineered.csv'
features.to_csv(output_path, index=False)
print(f"Features saved to: {output_path}")
print(f"Total features: {len(features.columns)}")
print()

Features saved to: features_engineered.csv
Total features: 96



In [14]:
# Display first few rows
print("\nSample of engineered features:")
display(features.head())

# Display basic statistics
print("\nBasic statistics:")
display(features.describe())


Sample of engineered features:


,game_id,game_date,season_id,team_id_home,team_id_away,home_win,fg_pct_home,fg_pct_away,fg3_pct_home,fg3_pct_away,...,reb_diff_L10,ast_diff_L10,tov_diff_L10,win_rate_diff_L10,fg_pct_diff_L20,fg3_pct_diff_L20,reb_diff_L20,ast_diff_L20,tov_diff_L20,win_rate_diff_L20
8,24600009,1946-11-05,21946,1610610028,1610610034,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,24600011,1946-11-07,21946,1610610032,1610610025,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
10,24600012,1946-11-07,21946,1610610034,1610612752,0,0.24,0.292,NaN,NaN,...,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,0.5
12,24600013,1946-11-08,21946,1610610035,1610610028,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,24600015,1946-11-09,21946,1610610032,1610610031,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0



Basic statistics:


,game_id,game_date,season_id,team_id_home,team_id_away,home_win,fg_pct_home,fg_pct_away,fg3_pct_home,fg3_pct_away,...,reb_diff_L10,ast_diff_L10,tov_diff_L10,win_rate_diff_L10,fg_pct_diff_L20,fg3_pct_diff_L20,reb_diff_L20,ast_diff_L20,tov_diff_L20,win_rate_diff_L20
count,6.557900e+04,65579,65579.000000,6.557900e+04,6.557900e+04,65579.000000,50126.000000,50127.000000,46569.000000,46680.000000,...,50237.000000,50056.000000,47411.000000,65519.000000,51348.000000,48237.000000,50629.000000,50316.000000,48413.000000,65519.000000
mean,2.584173e+07,1995-02-25 13:27:57.225941120,22943.779777,1.610318e+09,1.608943e+09,0.619070,0.467331,0.454889,0.346165,0.336644,...,1.659150,1.808856,-0.432136,0.237989,0.012186,0.009602,1.686192,1.811312,-0.455246,0.238091
min,1.050000e+07,1946-11-05 00:00:00,12005.000000,9.400000e+01,4.100000e+01,0.000000,0.140000,0.156000,0.000000,0.000000,...,-23.000000,-21.000000,-17.800000,-1.000000,-0.843583,-1.000000,-25.000000,-21.000000,-18.250000,-1.000000
25%,2.130052e+07,1982-12-21 00:00:00,21981.000000,1.610613e+09,1.610613e+09,0.000000,0.427000,0.416000,0.261000,0.250000,...,-1.000000,-0.666667,-2.000000,0.000000,-0.005250,-0.024500,-0.550000,-0.400000,-1.800000,0.050000
50%,2.630005e+07,1997-11-09 00:00:00,21997.000000,1.610613e+09,1.610613e+09,1.000000,0.467000,0.455000,0.348000,0.333000,...,1.600000,1.800000,-0.400000,0.200000,0.012650,0.008150,1.600000,1.800000,-0.400000,0.250000
75%,2.880066e+07,2010-04-09 00:00:00,22011.000000,1.610613e+09,1.610613e+09,1.000000,0.506000,0.494000,0.432000,0.424000,...,4.300000,4.200000,1.200000,0.400000,0.030300,0.041150,3.850000,4.000000,0.950000,0.400000
max,4.980009e+07,2023-06-12 00:00:00,42022.000000,1.610617e+09,1.610617e+09,1.000000,0.697000,3.556000,1.000000,1.000000,...,44.200000,25.500000,16.000000,1.000000,0.221500,1.000000,42.550000,21.500000,16.000000,1.000000
std,6.296284e+06,NaN,4992.976337,2.178513e+07,5.183682e+07,0.485619,0.059412,0.059185,0.151255,0.147273,...,4.091252,3.713201,2.511304,0.293587,0.029649,0.086683,3.568124,3.327251,2.294352,0.253147
